In [45]:
print("Hello World")

Hello World


In [46]:
from dotenv import load_dotenv
load_dotenv()

True

In [47]:
from dotenv import load_dotenv
load_dotenv()

True

In [49]:
import pandas as pd

# QA
inputs = [
    "What does RAG stand for and why is it used?",
    "What are the main stages of a typical RAG pipeline?",
    "What is the difference between fine-tuning and retrieval for injecting knowledge?",
    "What is maximal marginal relevance (MMR) used for?",
    "What are some common RAG failure modes?",
    "What is the difference between an agent and a simple question-answering system?",
    "What metrics do teams typically track once an AI system is in production?",
]

outputs = [
    "RAG stands for retrieval-augmented generation. It retrieves relevant documents or passages at query time and includes them in the prompt, which reduces hallucination and allows models to answer questions about private or recent data without retraining.",
    "A typical RAG pipeline loads documents, splits them into smaller chunks, converts each chunk into a vector embedding, stores the embeddings in a vector database, retrieves the most similar chunks to a query, and combines them with the question in a prompt sent to a language model.",
    "Fine-tuning is effective for teaching a model a specific style, format, or narrow skill, but is less effective for injecting large amounts of factual knowledge since that knowledge must be re-learned if facts change. Retrieval allows knowledge to be updated simply by changing the document store, without retraining.",
    "MMR balances relevance with diversity in retrieval results, preventing the retrieved set from containing many near-duplicate chunks.",
    "Common failure modes include retrieval failure (failing to find relevant documents), context dilution (too many irrelevant chunks retrieved), hallucination (the model ignoring provided context), and outdated information (a stale document store).",
    "Unlike simple question-answering systems, agents operate in a loop of observing the current state, deciding on an action, executing it, and observing the result before deciding on the next step, often using tools to take real-world actions.",
    "Teams typically track latency, cost per request, error rates, and user satisfaction signals, along with logging full context and outputs for debugging and offline evaluation.",
]

# Dataset
qa_pairs = [{"question": q, "answer": a} for q, a in zip(inputs, outputs)]
df = pd.DataFrame(qa_pairs)

# Write to csv
csv_path = "/Users/bandarusamanthuday/Desktop/llmops/data/goldens.csv"
df.to_csv(csv_path, index=False)

In [51]:
from langsmith import Client

client = Client()
dataset_name = "AIEngineeringSampleGoldens"  # or whatever name you're using

# Check if it already exists first
existing = list(client.list_datasets(dataset_name=dataset_name))

if existing:
    dataset = existing[0]
    print(f"Using existing dataset: {dataset.id}")
else:
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="Input and expected output pairs for the AI Engineering sample document",
    )
    client.create_examples(
        inputs=[{"question": q} for q in inputs],
        outputs=[{"answer": a} for a in outputs],
        dataset_id=dataset.id,
    )
    print(f"Created new dataset: {dataset.id}")

Using existing dataset: 8cf7efad-9f4b-4eb3-896c-76c803671adc


In [52]:
import sys
sys.path.append("/Users/bandarusamanthuday/Desktop/llmops")

from pathlib import Path
from multi_doc_chat.src.document_ingestion.data_ingestion import ChatIngestor
from multi_doc_chat.src.document_chat.retrieval import ConversationalRAG
import os


# Simple file adapter for local file paths
class LocalFileAdapter:
    """Adapter for local file paths to work with ChatIngestor."""
    def __init__(self, file_path: str):
        self.path = Path(file_path)
        self.name = self.path.name

    def getbuffer(self) -> bytes:
        return self.path.read_bytes()


def answer_ai_report_question(
    inputs: dict,
    data_path: str = "/Users/bandarusamanthuday/Desktop/llmops/data/The 2025 AI Engineering Report.txt",
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
    k: int = 5
) -> dict:
    """
    Answer questions about the AI engineering sample document using RAG.

    Args:
        inputs: Dictionary containing the question, e.g., {"question": "What is RAG?"}
        data_path: Path to the sample text file
        chunk_size: Size of text chunks for splitting
        chunk_overlap: Overlap between chunks
        k: Number of documents to retrieve

    Returns:
        Dictionary with the answer, e.g., {"answer": "RAG stands for..."}
    """
    try:
        # Extract question from inputs
        question = inputs.get("question", "")
        if not question:
            return {"answer": "No question provided"}

        # Check if file exists
        if not Path(data_path).exists():
            return {"answer": f"Data file not found: {data_path}"}

        # Create file adapter
        file_adapter = LocalFileAdapter(data_path)

        # Build index using ChatIngestor
        ingestor = ChatIngestor(
            temp_base="data",
            faiss_base="faiss_index",
            use_session_dirs=True
        )

        # Build retriever
        ingestor.built_retriver(
            uploaded_files=[file_adapter],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            k=k
        )

        # Get session ID and index path
        session_id = ingestor.session_id
        index_path = f"faiss_index/{session_id}"

        # Create RAG instance and load retriever
        rag = ConversationalRAG(session_id=session_id)
        rag.load_retriever_from_faiss(
            index_path=index_path,
            k=k,
            index_name=os.getenv("FAISS_INDEX_NAME", "index")
        )

        # Get answer
        answer = rag.invoke(question, chat_history=[])
        return {"answer": answer}

    except Exception as e:
        return {"answer": f"Error: {str(e)}"}

In [53]:
test_input = {"question": "What does RAG stand for and why is it used?"}
result = answer_ai_report_question(test_input)
print("Question:", test_input["question"])
print("\nAnswer:", result["answer"])

{"timestamp": "2026-09-20T14:28:02.062184Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-09-20T14:28:02.063806Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-09-20T14:28:02.064198Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_Su...", "GOOGLE_API_KEY": "AQ.Ab8..."}, "timestamp": "2026-09-20T14:28:02.064635Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-09-20T14:28:02.071763Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260920_195802_3b89ac2c", "temp_dir": "data/session_20260920_195802_3b89ac2c", "faiss_dir": "faiss_index/session_20260920_195802_3b89ac2c", "sessionized": true, "timestamp": "2026-09-20T14:28:02.073767Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "The 2025 AI Engineering Report.txt", "saved_as

Question: What does RAG stand for and why is it used?

Answer: RAG stands for Retrieval‑Augmented Generation. It is used to fetch relevant documents at query time, reducing hallucinations, enabling answers about private or recent data, and allowing information updates without retraining the model.


In [54]:
# Example: Test with all golden questions
print("Testing all questions from the dataset:\n")

for i, q in enumerate(inputs, 1):
    test_input = {"question": q}
    result = answer_ai_report_question(test_input)
    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}\n")
    print("-" * 80 + "\n")

{"timestamp": "2026-09-20T14:28:21.037076Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-09-20T14:28:21.037782Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-09-20T14:28:21.038292Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_Su...", "GOOGLE_API_KEY": "AQ.Ab8..."}, "timestamp": "2026-09-20T14:28:21.038783Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-09-20T14:28:21.042195Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260920_195821_d3dc66d3", "temp_dir": "data/session_20260920_195821_d3dc66d3", "faiss_dir": "faiss_index/session_20260920_195821_d3dc66d3", "sessionized": true, "timestamp": "2026-09-20T14:28:21.045105Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "The 2025 AI Engineering Report.txt", "saved_as

Testing all questions from the dataset:



HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fconfig_sentence_transformers.json=&etag=%22fd1b291129c607e5d49799f87cb219b27f98acdf%22 "HTTP/1.1 200 OK"
Loading SentenceTransformer model from se

Q1: What does RAG stand for and why is it used?
A1: RAG stands for Retrieval‑Augmented Generation. It is used to fetch relevant documents at query time, reducing hallucinations, enabling answers about private or recent data, and allowing information updates without retraining the model.

--------------------------------------------------------------------------------



HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fconfig_sentence_transformers.json=&etag=%22fd1b291129c607e5d49799f87cb219b27f98acdf%22 "HTTP/1.1 200 OK"
Loading SentenceTransformer model from se

...

Q5: What are some common RAG failure modes?
A5: Common RAG failure modes include retrieval failure (missing relevant documents), context dilution (irrelevant chunks crowding the answer), hallucination (the model ignoring or extrapolating beyond the retrieved context), and outdated information (answers based on stale data).

--------------------------------------------------------------------------------



HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fconfig_sentence_transformers.json=&etag=%22fd1b291129c607e5d49799f87cb219b27f98acdf%22 "HTTP/1.1 200 OK"
Loading SentenceTransformer model from se

Q6: What is the difference between an agent and a simple question-answering system?
A6: An agentic system gives the language model the ability to take actions, call external tools, and make multi‑step decisions in a loop of observing, deciding, and acting toward a goal. A simple question‑answering system only generates text responses without performing real‑world actions. Consequently, agents introduce additional safety and error‑recovery challenges that QA systems do not face.

--------------------------------------------------------------------------------



HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fconfig_sentence_transformers.json=&etag=%22fd1b291129c607e5d49799f87cb219b27f98acdf%22 "HTTP/1.1 200 OK"
Loading SentenceTransformer model from se

Q7: What metrics do teams typically track once an AI system is in production?
A7: Teams usually monitor latency, cost per request, error rates, and user‑satisfaction signals once an AI system is in production.

--------------------------------------------------------------------------------



In [55]:
from langsmith import evaluate

### built in evaluator


In [72]:
from langsmith import evaluate
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT
from langchain_groq import ChatGroq

# Judge model used to score correctness (reuse your existing Groq LLM setup)
judge_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

# Evaluator: replaces LangChainStringEvaluator("cot_qa")
correctness_evaluator = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    feedback_key="correctness",
    judge=judge_llm,
)

dataset_name = "AIEngineeringSampleGoldens"  # make sure this matches your actual LangSmith dataset name

# Run evaluation using our RAG function
experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=[correctness_evaluator],
    experiment_prefix="test-agenticAIReport-qa-rag",
    # Experiment metadata
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

View the evaluation results for experiment: 'test-agenticAIReport-qa-rag-0355edcf' at:
https://smith.langchain.com/o/ec38608e-0c6b-4831-9d66-001699ecdb4e/datasets/8cf7efad-9f4b-4eb3-896c-76c803671adc/compare?selectedSessions=b4c01f6a-f606-472a-9fde-af05927fab6f




0it [00:00, ?it/s]{"timestamp": "2026-09-20T18:26:49.385638Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-09-20T18:26:49.387530Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-09-20T18:26:49.387753Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_Su...", "GOOGLE_API_KEY": "AQ.Ab8..."}, "timestamp": "2026-09-20T18:26:49.389710Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-09-20T18:26:49.402080Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260920_235649_84573b3a", "temp_dir": "data/session_20260920_235649_84573b3a", "faiss_dir": "faiss_index/session_20260920_235649_84573b3a", "sessionized": true, "timestamp": "2026-09-20T18:26:49.406973Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "The 2025 AI Engineering Repo

# CUSTOM EVALUATOR ---- OUT OF SCOPE

In [69]:
from langsmith.schemas import Run, Example
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq


def correctness_evaluator(run: Run, example: Example) -> dict:
    """
    Custom LLM-as-a-Judge evaluator for correctness.

    Correctness means how well the actual model output matches the reference output
    in terms of factual accuracy, coverage, and meaning.

    Args:
        run: The Run object containing the actual outputs
        example: The Example object containing the expected outputs

    Returns:
        dict with 'score' (1 for correct, 0 for incorrect) and 'reasoning'
    """
    # Extract actual and expected outputs
    actual_output = run.outputs.get("answer", "")
    expected_output = example.outputs.get("answer", "")
    input_question = example.inputs.get("question", "")

    # Define the evaluation prompt
    eval_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an evaluator whose job is to judge correctness.

Correctness means how well the actual model output matches the reference output in terms of factual accuracy, coverage, and meaning.

- If the actual output matches the reference output semantically (even if wording differs), it should be marked correct.
- If the output misses key facts, introduces contradictions, or is factually incorrect, it should be marked incorrect.

Do not penalize for stylistic or formatting differences unless they change meaning."""),
        ("human", """<example>
<input>
{input}
</input>

<output>
Expected Output: {expected_output}

Actual Output: {actual_output}
</output>
</example>

Please grade the following agent run given the input, expected output, and actual output.
Focus only on correctness (semantic and factual alignment).

Respond with:
1. A brief reasoning (1-2 sentences)
2. A final verdict: either "CORRECT" or "INCORRECT"

Format your response as:
Reasoning: [your reasoning]
Verdict: [CORRECT or INCORRECT]""")
    ])

    try:
        # Initialize LLM (Groq, confirmed working elsewhere in this project)
        llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

        # Create chain and invoke
        chain = eval_prompt | llm

        print("Calling judge LLM...")
        response = chain.invoke({
            "input": input_question,
            "expected_output": expected_output,
            "actual_output": actual_output
        })
        print("Got response:", response.content[:200])
        response_text = response.content

        # Parse the response
        reasoning = ""
        verdict = ""
        for line in response_text.split('\n'):
            if line.startswith("Reasoning:"):
                reasoning = line.replace("Reasoning:", "").strip()
            elif line.startswith("Verdict:"):
                verdict = line.replace("Verdict:", "").strip()

        # Convert verdict to score (1 for correct, 0 for incorrect)
        score = 1 if "CORRECT" in verdict.upper() else 0

        return {
            "key": "correctness",
            "score": score,
            "reasoning": reasoning,
            "comment": f"Verdict: {verdict}"
        }

    except Exception as e:
        print("ERROR:", str(e))
        return {
            "key": "correctness",
            "score": 0,
            "reasoning": f"Error during evaluation: {str(e)}"
        }

In [70]:
experiment_results = evaluate(
    answer_ai_report_question,
    data="AIEngineeringSampleGoldens",  # your dataset name
    evaluators=[correctness_evaluator],
    experiment_prefix="test-ai-engineering-sample-v3",
    metadata={
        "variant": "RAG with FAISS and AI Engineering Sample",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

View the evaluation results for experiment: 'test-ai-engineering-sample-v3-5a20d765' at:
https://smith.langchain.com/o/ec38608e-0c6b-4831-9d66-001699ecdb4e/datasets/8cf7efad-9f4b-4eb3-896c-76c803671adc/compare?selectedSessions=a42a7d13-2b74-41fa-8bb7-21744d74da62




0it [00:00, ?it/s]{"timestamp": "2026-09-20T15:02:39.040440Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-09-20T15:02:39.041499Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-09-20T15:02:39.041919Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_Su...", "GOOGLE_API_KEY": "AQ.Ab8..."}, "timestamp": "2026-09-20T15:02:39.042425Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-09-20T15:02:39.049722Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260920_203239_818aef59", "temp_dir": "data/session_20260920_203239_818aef59", "faiss_dir": "faiss_index/session_20260920_203239_818aef59", "sessionized": true, "timestamp": "2026-09-20T15:02:39.051506Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "The 2025 AI Engineering Repo

Calling judge LLM...


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Error running evaluator <DynamicRunEvaluator correctness_evaluator> on run 01a0bf57-36bc-7500-9068-c5119d445f92: ValueError("Expected an EvaluationResult object, or dict with a metric 'key' and optional 'score'; got {'key': 'correctness', 'score': 1, 'reasoning': 'The actual output conveys the same meaning as the expected output, correctly defining RAG and its purpose without missing or altering any key facts.', 'comment': 'Verdict: CORRECT'}")
Traceback (most recent call last):
  File "/Users/bandarusamanthuday/Desktop/llmops/.venv/lib/python3.10/site-packages/langsmith/evaluation/evaluator.py", line 286, in _coerce_evaluation_result
    return EvaluationResult(**{"source_run_id": source_run_id, **result})
  File "/Users/bandarusamanthuday/Desktop/llmops/.venv/lib/python3.10/site-packages/pydantic/main.py", line 263, in __init__
    validated_self = self.__pydantic_validator__.validate_python(data, se

Got response: Reasoning: The actual output conveys the same meaning as the expected output, correctly defining RAG and its purpose without missing or altering any key facts.
Verdict: CORRECT


...

HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fconfig_sentence_transformers.json=&etag=%22fd1b291129c607e5d49799f87cb219b27f98acdf%22 "HTTP/1.1 200 OK"
Loading SentenceTransformer model from se

Calling judge LLM...


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Error running evaluator <DynamicRunEvaluator correctness_evaluator> on run 01a0bf58-a01d-7603-a181-68abf884bb78: ValueError("Expected an EvaluationResult object, or dict with a metric 'key' and optional 'score'; got {'key': 'correctness', 'score': 1, 'reasoning': 'The actual output conveys the same factual information as the expected output, describing MMR as balancing relevance and diversity to avoid near‑duplicate chunks, just with slightly different wording.', 'comment': 'Verdict: CORRECT'}")
Traceback (most recent call last):
  File "/Users/bandarusamanthuday/Desktop/llmops/.venv/lib/python3.10/site-packages/langsmith/evaluation/evaluator.py", line 286, in _coerce_evaluation_result
    return EvaluationResult(**{"source_run_id": source_run_id, **result})
  File "/Users/bandarusamanthuday/Desktop/llmops/.venv/lib/python3.10/site-packages/pydantic/main.py", line 263, in __init__
    validated_self = 

Got response: Reasoning: The actual output conveys the same factual information as the expected output, describing MMR as balancing relevance and diversity to avoid near‑duplicate chunks, just with slightly differe


7it [01:46, 15.26s/it]


| **Aspect** | **OpenEvals Version (This One)** | **Custom Gemini Version (Earlier)** |
|---|---|---|
| **Judge logic** | Pre-built by `openevals` (`create_llm_as_judge` + `CORRECTNESS_PROMPT`) | You write and control the entire prompt yourself |
| **Prompt** | Maintained/tuned by the `openevals` library authors, hidden from you | Fully visible and editable — you crafted the exact grading criteria and output format |
| **Judge model** | Groq (`openai/gpt-oss-120b`) — fast, cheap, already confirmed working for you | Gemini (`gemini-2.5-pro`) — untested/unconfirmed for your API key |
| **Output parsing** | Handled internally by `openevals` — you don't see or control how it extracts the score | You manually parse `Reasoning:` / `Verdict:` text lines yourself — more fragile, but fully transparent |
| **Dependencies** | Requires installing and trusting an extra third-party package (`openevals`) | Zero extra dependencies — just `langchain_core` and `langchain_google_genai`, which you already have |
| **Maintenance risk** | Tied to `openevals`'s own versioning/updates | Tied to nothing but your own code — won't break from an upstream library change |
| **Feedback fields returned** | Whatever `openevals` decides to return (likely `score` + some reasoning) | You define exactly what comes back: `score`, `reasoning`, `comment` |